In [1]:
from sentence_transformers import SentenceTransformer

# Load the local model (it will download about 80MB the first time you run this)
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# The legal text we want to mathematically encode
text_to_embed = "La période d'essai pour un CDI est de 3 mois pour les cadres."

# Generate the vector locally on your ThinkPad's CPU
vector = embedding_model.encode(text_to_embed)

print(f"Dimension of the vector (Length): {len(vector)}")
print(f"First 5 numbers of the embedding:\n{vector[:5]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dimension of the vector (Length): 384
First 5 numbers of the embedding:
[-0.17237613 -0.01570499  0.3107673  -0.46348062  0.00488091]


In [2]:
from elasticsearch import Elasticsearch

# 1. Increase timeout to 60 seconds to give Docker enough time
es_client = Elasticsearch(
    'http://localhost:9200',
    request_timeout=60
)

index_name = "juris_maroc_vectors"

# 2. Separate settings and mappings (fixes the DeprecationWarning)
settings = {
    "number_of_shards": 1,
    "number_of_replicas": 0
}

mappings = {
    "properties": {
        "text": {"type": "text"},
        "section": {"type": "keyword"},
        "text_vector": {
            "type": "dense_vector",
            "dims": 384,
            "index": True,
            "similarity": "cosine"
        }
    }
}

# Delete the index if it already exists
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

# 3. Pass settings and mappings directly instead of using 'body'
es_client.indices.create(
    index=index_name,
    settings=settings,
    mappings=mappings
)

print(f"Index '{index_name}' created successfully with Vector Search capabilities.")

Index 'juris_maroc_vectors' created successfully with Vector Search capabilities.


In [6]:
import requests

# 1. Check Elasticsearch health and disk usage
health = requests.get("http://localhost:9200/_cluster/health").json()
print("Cluster Status:", health.get("status"))  # Should be 'green' or 'yellow'. If 'red', shards are stuck!

# 2. Unlock Elasticsearch if your hard drive is >85% full (Very common cause of timeouts)
unlock_response = requests.put(
    "http://localhost:9200/_cluster/settings",
    json={
        "transient": {
            "cluster.routing.allocation.disk.threshold_enabled": False,
            "index.blocks.read_only_allow_delete": None
        }
    }
)
print("Unlock response:", unlock_response.status_code)

# 3. Force Elasticsearch to retry assigning any stuck shards
requests.post("http://localhost:9200/_cluster/reroute?retry_failed=true")

Cluster Status: red
Unlock response: 400


<Response [200]>

In [8]:
import json
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
from tqdm.auto import tqdm

# 1. Connect to Elasticsearch WITH the 60-second timeout!
es_client = Elasticsearch(
    'http://localhost:9200',
    request_timeout=60
)

index_name = "juris_maroc_vectors"
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

# Make sure the index exists with the right mappings before inserting
if not es_client.indices.exists(index=index_name):
    es_client.indices.create(
        index=index_name,
        settings={"number_of_shards": 1, "number_of_replicas": 0},
        mappings={
            "properties": {
                "text": {"type": "text"},
                "article_number": {"type": "keyword"},
                "id": {"type": "keyword"},
                "source": {"type": "keyword"},
                "text_vector": {
                    "type": "dense_vector",
                    "dims": 384,
                    "index": True,
                    "similarity": "cosine"
                }
            }
        }
    )

# 2. Load the Moroccan Labor Code data
with open('../LLM_RAG/labor_code_chunks.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

# 3. Loop, Embed, and Index
for doc in tqdm(documents, desc="Embedding and Indexing Laws"):
    vector = embedding_model.encode(doc['content'])

    es_doc = {
        "text": doc['content'],
        "article_number": doc.get('article_number', 'N/A'),
        "id": doc.get('id', ''),
        "source": doc.get('source', ''),
        "text_vector": vector.tolist()
    }

    # Inject into the database
    es_client.index(index=index_name, document=es_doc)

print("✅ All legal documents are successfully embedded and indexed!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding and Indexing Laws:   0%|          | 0/587 [00:00<?, ?it/s]

ApiError: ApiError(503, 'unavailable_shards_exception', '[juris_maroc_vectors][0] primary shard is not active Timeout: [1m], request: [BulkShardRequest [[juris_maroc_vectors][0]] containing [index {[juris_maroc_vectors][aLtw_6EBdwpBSdvesgrP], source[n/a, actual length: [8.5kb], max length: 2kb]}]]')

In [9]:
# Look at the keys of the very first chunk in your JSON
print(documents[0].keys())

dict_keys(['id', 'article_number', 'content', 'source', 'language'])


In [11]:
# 1. The user asks a natural question
user_query = "Comment je peux démissionner de mon travail ?"

# 2. Translate the question into a 384-dimensional vector arrow
query_vector = embedding_model.encode(user_query).tolist()

# 3. Build the Elasticsearch KNN payload
search_query = {
    "knn": {
        "field": "text_vector",
        "query_vector": query_vector,
        "k": 3,               # Return the top 3 closest arrows (matches)
        "num_candidates": 100 # Optimization: Only calculate exact angles for the 100 closest candidates
    },
    "_source": ["article_number", "text"] # Only return these fields to save bandwidth
}

# 4. Execute the search against your local container
response = es_client.search(
    index=index_name,
    knn=search_query["knn"],
    _source=search_query["_source"]
)

# 5. Display the matching articles and their Cosine Similarity scores
print(f"Question: '{user_query}'\n" + "-"*50)
for hit in response['hits']['hits']:
    score = hit['_score']
    article = hit['_source'].get('article_number', 'N/A')
    text = hit['_source']['text']

    print(f"Score de similarité: {score}")
    print(f"Article: {article}")
    print(f"Extrait: {text[:200]}...\n")

Question: 'Comment je peux démissionner de mon travail ?'
--------------------------------------------------
Score de similarité: 0.77681625
Article: 218
Extrait: Article 218 
Il peut être décidé que le jour férié soit rémunéré comme temps de 
travail effectif....

Score de similarité: 0.7731632
Article: 231
Extrait: Article 231 
Sauf dispositions plus favorables du contrat de travail, de la 
convention collective de travail, du règlement intérieur ou des usages, 
tout salarié a droit, après six mois de service co...

Score de similarité: 0.76560414
Article: 352
Extrait: Article 352 
Sont considérées comme périodes de travail effectif et ne peuvent 
être déduites de la durée des services entrant en ligne de compte pour 
l'attribution de la prime d'ancienneté : 
1. les...



In [11]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
import json
from tqdm.auto import tqdm

# 1. Force the notebook to use the Multilingual model
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris_maroc_vectors"

# 2. DESTROY the old database to clear out the English vectors
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

# Recreate the clean index
es_client.indices.create(index=index_name, body={
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "article_number": {"type": "keyword"},
            "text_vector": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"}
        }
    }
})

# 3. Re-ingest the data using the new French-aware model
with open('../LLM_RAG/labor_code_chunks.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

for doc in tqdm(documents, desc="Embedding Multilingual Laws"):
    es_doc = {
        "text": doc['content'],
        "article_number": doc.get('article_number', 'N/A'),
        "text_vector": embedding_model.encode(doc['content']).tolist()
    }
    es_client.index(index=index_name, document=es_doc)

# 4. Search again (Ensuring the query is embedded with the new model)
user_query = "Comment je peux démissionner de mon travail ?"
query_vector = embedding_model.encode(user_query).tolist()

response = es_client.search(
    index=index_name,
    knn={"field": "text_vector", "query_vector": query_vector, "k": 3, "num_candidates": 100},
    _source=["article_number", "text"]
)

print(f"\nQuestion: '{user_query}'\n" + "-"*50)
for hit in response['hits']['hits']:
    print(f"Score: {hit['_score']}")
    print(f"Article: {hit['_source'].get('article_number', 'N/A')}")
    print(f"Extrait: {hit['_source']['text'][:200]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_5772/347768108.py:16: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body={


Embedding Multilingual Laws:   0%|          | 0/587 [00:00<?, ?it/s]


Question: 'Comment je peux démissionner de mon travail ?'
--------------------------------------------------
Score: 0.77095264
Article: 544
Extrait: Article 544  
L'employeur est tenu de verser aux salariés qui ont cessé de 
travailler, en raison de la fermeture prévue au deuxième alinéa de 
l'article 543 ci-dessus, une rémunération pour la périod...

Score: 0.76188976
Article: 510
Extrait: Article 510  
Le salarié tenu de quitter son emploi en vue d'accomplir le service 
militaire83 a le droit de reprendre son poste ou, à défaut, un poste de la 
même profession, dans l'entreprise, à la ...

Score: 0.7589003
Article: 263
Extrait: Article 263 
Il est interdit à tout salarié bénéficiaire d'un congé annuel payé 
d'exécuter des travaux rémunérés pendant son congé. 
 
  
 
Section VII : Dispositions relatives au salarié travaillant...



In [5]:
from rag_helper import RAGBase
from sentence_transformers import SentenceTransformer
from google.genai import types

class ElasticVectorRAG(RAGBase):
    def __init__(self, es_client, index_name, llm_client, instructions):
        super().__init__(index=None, llm_client=llm_client, instructions=instructions)
        self.es_client = es_client
        self.index_name = index_name
        self.embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def search(self, query, num_results=5):
        query_vector = self.embedding_model.encode(query).tolist()

        response = self.es_client.search(
            index=self.index_name,
            knn={
                "field": "text_vector",
                "query_vector": query_vector,
                "k": num_results,
                "num_candidates": 100
            },
            _source=["article_number", "text"]
        )

        results = []
        for hit in response['hits']['hits']:
            doc = hit['_source']
            results.append({
                "article_number": doc.get("article_number", "N/A"),
                "content": doc.get("text", "")
            })
        return results

    # 👇 OVERRIDE THE LLM METHOD TO FIX THE GEMINI CLIENTERROR
    def llm(self, prompt):
        try:
            response = self.llm_client.models.generate_content(
                model="gemini-3.1-flash-lite",  # Uses Google's current active model
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=self.instructions,
                    temperature=0.3,
                )
            )
            return response.text
        except Exception as e:
            print(f"❌ Exact Google Gemini Error: {e}")
            raise e

In [6]:
# 1. Initialize your LLM client (assuming you have this from your other cells)
from elasticsearch import Elasticsearch
es_client = Elasticsearch('http://localhost:9200')
import os
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
juris_instructions = "Tu es un assistant juridique expert..."

# 2. Instantiate your new integrated Vector RAG pipeline
vector_rag_system = ElasticVectorRAG(
    es_client=es_client,
    index_name="juris_maroc_vectors",
    llm_client=gemini_client,
    instructions=juris_instructions
)

# 3. Ask a natural question (This triggers the search AND the LLM generation)
user_query = "Quelles sont les conditions de la démission ?"

# Depending on how you wrote the rest of RAGBase, call your main execution method:
# (In the standard Zoomcamp RAGBase, this is usually called by directly calling the object or an answer/ask method)
# Call the method explicitly instead of treating the object like a function
final_answer = vector_rag_system.rag(user_query)

print("⚖️ RÉPONSE JURIS MAROC :")
print(final_answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⚖️ RÉPONSE JURIS MAROC :
En tant qu'assistant juridique, et sur la base des textes que vous avez fournis, voici les éléments relatifs à la démission :

Le contexte juridique fourni est limité concernant les conditions de fond de la démission (telles que le préavis ou la forme de la lettre). Cependant, il apporte une précision procédurale importante :

*   **Obligation de notification à l'inspection du travail :** Selon l'article 64, une copie de la lettre de démission doit être adressée à l'agent chargé de l'inspection du travail.

Par ailleurs, les textes mentionnent la démission dans deux autres cas spécifiques qui ne concernent pas la procédure générale de rupture du contrat de travail :

1.  **Délégués des salariés (Article 435) :** La démission est citée comme l'une des causes mettant fin aux fonctions de délégué des salariés (au même titre que le décès, le retrait de confiance, l'âge de la retraite ou la rupture du contrat de travail).

**Note importante :** Les articles fournis 

In [1]:
import json
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

# 1. Setup
es_client = Elasticsearch('http://localhost:9200')
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
index_name = "juris_maroc_vectors"

# 2. Create the index
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

es_client.indices.create(index=index_name, body={
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "article_number": {"type": "keyword"},
            "text_vector": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"}
        }
    }
})

# 3. Load and ingest the data
with open('labor_code_chunks.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

for doc in tqdm(documents, desc="Re-indexing Laws"):
    es_doc = {
        "text": doc['content'],
        "article_number": doc.get('article_number', 'N/A'),
        "text_vector": embedding_model.encode(doc['content']).tolist()
    }
    es_client.index(index=index_name, document=es_doc)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_4145/2376379004.py:15: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body={


Re-indexing Laws:   0%|          | 0/587 [00:00<?, ?it/s]

In [3]:
from rag_helper import RAGBase
from sentence_transformers import SentenceTransformer
from google.genai import types

class ElasticHybridRAG(RAGBase):
    def __init__(self, es_client, index_name, llm_client, instructions):
        super().__init__(index=None, llm_client=llm_client, instructions=instructions)
        self.es_client = es_client
        self.index_name = index_name
        self.embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def search(self, query, num_results=5, k_rrf=60):
        # --- 1. KEYWORD SEARCH (BM25) ---
        bm25_response = self.es_client.search(
            index=self.index_name,
            query={
                "multi_match": {
                    "query": query,
                    "fields": ["text", "article_number^2"] # Boosts exact article numbers!
                }
            },
            size=20,
            _source=["article_number", "text"]
        )
        bm25_hits = bm25_response['hits']['hits']

        # --- 2. VECTOR SEARCH (k-NN) ---
        query_vector = self.embedding_model.encode(query).tolist()
        knn_response = self.es_client.search(
            index=self.index_name,
            knn={
                "field": "text_vector",
                "query_vector": query_vector,
                "k": 20,
                "num_candidates": 100
            },
            _source=["article_number", "text"]
        )
        knn_hits = knn_response['hits']['hits']

        # --- 3. FUSION: RRF (Reciprocal Rank Fusion) ---
        # Formula: RRF_Score = 1 / (rank + 60)
        rrf_scores = {}
        doc_store = {}

        # Score the Keyword (BM25) results by their rank (1st, 2nd, 3rd...)
        for rank, hit in enumerate(bm25_hits):
            doc_id = hit['_id']
            doc_store[doc_id] = hit['_source']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rank + 1 + k_rrf))

        # Add the Vector (k-NN) scores by their rank
        for rank, hit in enumerate(knn_hits):
            doc_id = hit['_id']
            doc_store[doc_id] = hit['_source']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rank + 1 + k_rrf))

        # Sort documents by highest combined RRF Fusion score and take Top 5
        sorted_ids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:num_results]

        # --- 4. FORMAT FOR RAGBASE ---
        fused_results = []
        for doc_id in sorted_ids:
            doc = doc_store[doc_id]
            fused_results.append({
                "article_number": doc.get("article_number", "N/A"),
                "content": doc.get("text", ""),
                "fusion_score": round(rrf_scores[doc_id], 5)
            })

        return fused_results

    def llm(self, prompt):
        response = self.llm_client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=self.instructions,
                temperature=0.3,
            )
        )
        return response.text

In [12]:
hybrid_rag = ElasticHybridRAG(
    es_client=es_client,
    index_name="juris_maroc_vectors",
    llm_client=gemini_client,
    instructions=juris_instructions
)

# Ask a question that mixes natural language + legal concepts
answer = hybrid_rag.rag("Quelles sont les conditions de la démission et le délai de préavis ?")
print(answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

En tant qu'expert juridique, voici l'analyse des conditions de la démission et des règles relatives au délai de préavis, basée sur les textes fournis :

### 1. Conditions de la démission
Pour qu'une démission soit valide, elle doit respecter les conditions suivantes :
*   **Volonté du salarié :** La démission doit émaner de la volonté expresse du salarié (Article 34).
*   **Formalisme :** La lettre de démission doit impérativement porter une **signature légalisée** par l'autorité compétente (Article 34).
*   **Notification :** Une copie de la lettre de démission doit être adressée à l'agent chargé de l'inspection du travail (Article 64).

### 2. Le délai de préavis
Le salarié qui démissionne est tenu de respecter un délai de préavis (Article 34). Les règles encadrant ce délai sont les suivantes :

*   **Sources du délai :** La durée du préavis est déterminée par les textes législatifs et réglementaires, le contrat de travail, la convention collective, le règlement intérieur ou les usag

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from rag_helper import RAGBase

load_dotenv(override=True)
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

rag_system = RAGBase(index=None, llm_client=gemini_client)

answer = rag_system.rag("Quelles sont les conditions de la démission et le délai de préavis  ?")
print(answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

D'après les articles fournis, voici les éléments concernant la démission et le délai de préavis :

**Conditions de la démission :**
*   Le contrat de travail à durée indéterminée peut cesser par la volonté du salarié au moyen d'une démission portant sa signature légalisée par l'autorité compétente (Article 34).
*   Une copie de la lettre de démission doit être adressée à l'agent chargé de l'inspection du travail (Article 64).

**Délai de préavis :**
*   Le salarié est tenu de respecter le délai de préavis (Article 34).
*   La rupture unilatérale du contrat est subordonnée, en l'absence de faute grave de l'autre partie, au respect de ce délai (Article 43).
*   Le délai et la durée du préavis sont réglementés par les textes législatifs et réglementaires, le contrat de travail, la convention collective, le règlement intérieur ou les usages (Article 43).
*   Toute clause fixant un délai de préavis inférieur à la durée fixée par les textes législatifs ou réglementaires est nulle. Dans tous 

In [6]:
print(list(rag_system.es_client.indices.get_alias(index="*").keys()))

['juris_maroc_vectors']
